# Module 09 Lab — Observability, Cost, and Reliability

**Course:** AI Agents Teaching Platform  
**Module:** 09 — Observability, Cost & Reliability  
**Estimated time:** 2–3 hours

---

### What you'll build

An observable, cost-tracked, and circuit-breaking agent:

1. **Tracing** — Langfuse `@observe()` decorators on all agent functions
2. **Cost audit** — token cost accumulator with per-model pricing
3. **SLO enforcement** — latency and error-rate checks with `sys.exit(1)` on breach
4. **Circuit breaker** — `pybreaker.CircuitBreaker(fail_max=3)` with graceful fallback

### Prerequisites

- A free Langfuse Cloud account at https://cloud.langfuse.com (takes ~2 minutes)
- Your `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` from the Langfuse dashboard

In [ ]:
%pip install -q langfuse litellm pybreaker pyyaml

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"]    = getpass("Anthropic API key: ")
os.environ["LANGFUSE_PUBLIC_KEY"]  = getpass("Langfuse public key: ")
os.environ["LANGFUSE_SECRET_KEY"]  = getpass("Langfuse secret key: ")
os.environ["LANGFUSE_HOST"]        = "https://cloud.langfuse.com"

print("Keys set ✓")
print("Open https://cloud.langfuse.com → Traces after running the agent")

## Step 1 — Shared setup: model config + cost tracking

In [ ]:
import time
import json
import sys
import litellm

litellm.drop_params = True

MODEL = "claude-haiku-4-5-20251001"

# Per-million-token pricing (adjust for your provider)
COST_PER_M_INPUT  = 1.00   # USD per million input tokens
COST_PER_M_OUTPUT = 5.00   # USD per million output tokens

COST_AUDIT = {"calls": 0, "input_tokens": 0, "output_tokens": 0, "total_usd": 0.0, "latencies_ms": []}

def tracked_call(messages: list, max_tokens: int = 512) -> str:
    """LiteLLM call that accumulates token cost and latency."""
    start = time.perf_counter()
    response = litellm.completion(model=MODEL, messages=messages, max_tokens=max_tokens)
    elapsed_ms = (time.perf_counter() - start) * 1000

    inp = response.usage.prompt_tokens
    out = response.usage.completion_tokens
    cost = (inp * COST_PER_M_INPUT + out * COST_PER_M_OUTPUT) / 1e6

    COST_AUDIT["calls"]         += 1
    COST_AUDIT["input_tokens"]  += inp
    COST_AUDIT["output_tokens"] += out
    COST_AUDIT["total_usd"]     += cost
    COST_AUDIT["latencies_ms"].append(elapsed_ms)

    return response.choices[0].message.content

print("Cost tracking setup ✓")

## Step 2 — Instrument with Langfuse `@observe()` (TODO)

Langfuse traces every call decorated with `@observe()`. Each decorator creates a span
in the Langfuse UI so you can drill into individual function calls.

**TODO:** Add `@observe()` to the two functions below. Then run Step 5 and verify traces appear in the Langfuse UI.

In [ ]:
from langfuse.decorators import observe, langfuse_context

# TODO: add @observe() decorator to both functions below

def research(question: str) -> str:
    result = tracked_call([
        {"role": "system", "content": "You are a research assistant. Be concise."},
        {"role": "user",   "content": f"Research: {question}"},
    ])
    langfuse_context.update_current_observation(
        name="research",
        input={"question": question},
        output=result,
    )
    return result


def summarise(content: str) -> str:
    result = tracked_call([
        {"role": "user", "content": f"Summarise in 2 sentences:\n{content}"},
    ])
    langfuse_context.update_current_observation(
        name="summarise",
        input={"content_length": len(content)},
        output=result,
    )
    return result


print("Agent functions defined — add @observe() above")

<details>
<summary>💡 Stuck? Reveal the @observe() solution</summary>

```python
from langfuse.decorators import observe, langfuse_context

@observe()
def research(question: str) -> str:
    ...

@observe()
def summarise(content: str) -> str:
    ...
```

After running the agent, go to https://cloud.langfuse.com → Traces.
You should see a trace with two nested spans: `research` → `summarise`.

</details>

## Step 3 — Circuit breaker (TODO)

The circuit breaker prevents cascading failures: after `fail_max=3` consecutive failures,
it opens and routes all calls to the fallback for `reset_timeout=30` seconds.

**TODO:** Implement `run_with_fallback()` — wrap `research()` in the circuit breaker.
If the circuit opens (raises `CircuitBreakerError`), return the fallback response.

In [ ]:
import pybreaker

breaker = pybreaker.CircuitBreaker(fail_max=3, reset_timeout=30)

@breaker
def _protected_research(question: str) -> str:
    return research(question)


FALLBACK_RESPONSE = (
    "[FALLBACK] Research service is temporarily unavailable. "
    "Providing cached response: The topic is complex and warrants further investigation. "
    "Please retry after 30 seconds."
)


def run_with_fallback(question: str) -> str:
    """TODO: call _protected_research(question) inside a try/except.
    Catch pybreaker.CircuitBreakerError and return FALLBACK_RESPONSE.
    Catch any other Exception, re-raise it (don't swallow real errors).
    """
    pass


print("Circuit breaker defined — implement run_with_fallback above")

<details>
<summary>💡 Stuck? Reveal the circuit breaker solution</summary>

```python
def run_with_fallback(question: str) -> str:
    try:
        return _protected_research(question)
    except pybreaker.CircuitBreakerError:
        print("[CIRCUIT OPEN] Returning fallback response")
        return FALLBACK_RESPONSE
```

Note: only `CircuitBreakerError` gets the fallback. Other exceptions (network errors, auth failures)
propagate normally so you notice real problems rather than silently masking them.

</details>

## Step 4 — SLO definitions and enforcement (TODO)

SLOs (Service Level Objectives) define what 'healthy' looks like. When they're violated,
the regression runner exits with code 1 (CI-friendly).

**TODO:** Fill in the `check_slos()` function body.

In [ ]:
import statistics

SLO_DEFINITIONS = {
    "p50_latency_ms":       2000,   # p50 must be under 2s
    "p95_latency_ms":       5000,   # p95 must be under 5s
    "max_cost_per_run_usd": 0.10,   # total run cost cap
    "error_rate_pct":       10.0,   # <10% error rate
}


def check_slos(audit: dict, error_count: int, total_calls: int) -> dict:
    """TODO: compute latency percentiles and cost, check against SLO_DEFINITIONS.
    Return {"passed": bool, "violations": [{"slo": name, "actual": val, "limit": limit}, ...]}.
    
    Hints:
    - latencies = sorted(audit["latencies_ms"])
    - p50 = statistics.median(latencies)
    - p95 index = int(0.95 * len(latencies))
    - error_rate = (error_count / total_calls * 100) if total_calls > 0 else 0
    """
    pass


print("SLO checker defined — implement check_slos above")

<details>
<summary>💡 Stuck? Reveal the SLO checker solution</summary>

```python
def check_slos(audit: dict, error_count: int, total_calls: int) -> dict:
    latencies = sorted(audit["latencies_ms"])
    if not latencies:
        return {"passed": True, "violations": []}

    p50 = statistics.median(latencies)
    p95 = latencies[min(int(0.95 * len(latencies)), len(latencies) - 1)]
    error_rate = (error_count / total_calls * 100) if total_calls > 0 else 0

    checks = [
        ("p50_latency_ms",       p50,                   SLO_DEFINITIONS["p50_latency_ms"]),
        ("p95_latency_ms",       p95,                   SLO_DEFINITIONS["p95_latency_ms"]),
        ("max_cost_per_run_usd", audit["total_usd"],    SLO_DEFINITIONS["max_cost_per_run_usd"]),
        ("error_rate_pct",       error_rate,            SLO_DEFINITIONS["error_rate_pct"]),
    ]
    violations = [
        {"slo": name, "actual": actual, "limit": limit}
        for name, actual, limit in checks if actual > limit
    ]
    return {"passed": not violations, "violations": violations}
```

</details>

## Step 5 — Run the observable agent

In [ ]:
# Reset cost audit
for k in COST_AUDIT:
    COST_AUDIT[k] = 0 if isinstance(COST_AUDIT[k], (int, float)) else []

QUESTIONS = [
    "What is retrieval-augmented generation?",
    "How does speculative decoding improve LLM inference speed?",
    "What are the main tradeoffs in model quantisation?",
]

error_count = 0
for q in QUESTIONS:
    print(f"\nQ: {q}")
    try:
        raw = run_with_fallback(q)
        summary = summarise(raw)
        print(f"Summary: {summary[:150]}...")
    except Exception as e:
        error_count += 1
        print(f"ERROR: {e}")

print("\n--- Cost Audit ---")
print(f"  Calls:         {COST_AUDIT['calls']}")
print(f"  Input tokens:  {COST_AUDIT['input_tokens']:,}")
print(f"  Output tokens: {COST_AUDIT['output_tokens']:,}")
print(f"  Total cost:    ${COST_AUDIT['total_usd']:.4f}")
if COST_AUDIT["latencies_ms"]:
    print(f"  p50 latency:   {statistics.median(COST_AUDIT['latencies_ms']):.0f}ms")

print("\n--- SLO Check ---")
result = check_slos(COST_AUDIT, error_count, len(QUESTIONS))
print(f"  Status: {'PASS ✓' if result['passed'] else 'FAIL ✗'}")
for v in result.get("violations", []):
    print(f"  VIOLATION: {v['slo']} = {v['actual']:.2f} (limit: {v['limit']})")

if not result["passed"]:
    print("\nSLO violations detected — in CI this would sys.exit(1)")
    # sys.exit(1)  # uncomment to enforce in CI

print("\n✓ Check traces at https://cloud.langfuse.com → Traces")

## Step 6 — Test the circuit breaker

In [ ]:
# Simulate 3 failures to open the circuit
import pybreaker

print("Simulating 3 consecutive failures...")
@breaker
def _always_fails(question: str) -> str:
    raise RuntimeError("Simulated LLM failure")

for i in range(3):
    try:
        _always_fails("test")
    except RuntimeError as e:
        print(f"  Failure {i+1}: {e} (circuit state: {breaker.current_state})")
    except pybreaker.CircuitBreakerError:
        print(f"  Circuit already OPEN — call blocked")

print(f"\nCircuit state: {breaker.current_state}")

# Now run_with_fallback should return fallback
fallback = run_with_fallback("What is attention?")
print(f"\nFallback response: {fallback[:80]}...")
assert "FALLBACK" in fallback, "Circuit should be open — check run_with_fallback"
print("Circuit breaker test passed ✓")

## Step 7 — Experiments

### Experiment A: LLM-as-judge in Langfuse
In the Langfuse UI, navigate to a trace and add an LLM-as-judge evaluator.
Configure it to score output quality 1–5. Run the agent again and check the scores.

### Experiment B: Custom score
After `run_with_fallback()`, use `langfuse_context.score_current_trace()` to add
a `word_count` score (number of words in the output). This appears in the Langfuse UI.

### Experiment C: Tighten the SLO
Change `p95_latency_ms` to `1000`. Does your agent breach it? What would you do in production?

### Experiment D: Adjust circuit breaker sensitivity
Try `fail_max=1` (opens after one failure) vs `fail_max=10`. What are the tradeoffs?
When would you use each in production?

In [ ]:
# Scratch cell — use for experiments


## Step 8 — Reflection

1. What is the difference between tracing (Langfuse) and logging (Python `logging`)? When do you need each?
2. Why does `check_slos` use `sys.exit(1)` rather than raising an exception?
3. The circuit breaker opens after 3 failures in 30 seconds. What are the failure modes of this design?
4. You have token cost data for 100 production calls. What would you plot first to identify cost anomalies?

*(Double-click to edit)*

1. 
2. 
3. 
4. 